In [1]:
import torch
from mininn import (
    Conv2D, ReLU, MaxPool2D, Flatten,
    Linear, Sequential, ResidualBlock,Adam
)
from mininn.training import train_one_epoch, evaluate
from torchvision import datasets, transforms
from copy import copy
from torch.utils.data import DataLoader, random_split, Subset
from mininn.checkpoint import save_checkpoint, load_checkpoint

import importlib
import mininn.optim
importlib.reload(mininn.optim)
from mininn.optim import Adam


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
g = torch.Generator(device=device).manual_seed(42)

data_root = "../data_1"
batch_size = 64

In [9]:
model = Sequential([
    Conv2D(3, 8, 3, padding=1, device=device, generator=g),
    ReLU(),
    MaxPool2D(2, 2),

    Conv2D(8, 16, 3, padding=1, device=device, generator=g),
    ReLU(),
    MaxPool2D(2, 2),

    ResidualBlock(16, device=device, generator=g),
    Flatten(),
    Linear(1024, 10, device=device, generator=g),
])

images = torch.randn(
    (2, 3, 32, 32), device=device, generator=g
)
labels = torch.tensor([0, 1], device=device)

model.train()
logits = model(images)
loss = torch.nn.functional.cross_entropy(logits, labels)
loss.backward()

print("Output shape:", logits.shape)
print("Loss:", loss.item())
print("All parameter gradients present:", all(
    p.grad is not None for p in model.parameters()
))
print("All parameter gradients finite:", all(
    p.grad is not None and torch.isfinite(p.grad).all().item()
    for p in model.parameters()
))

Output shape: torch.Size([2, 10])
Loss: 7.869256973266602
All parameter gradients present: True
All parameter gradients finite: True


In [3]:
w = torch.tensor([2.0], device=device, requires_grad=True)
optimizer = Adam([w], learning_rate=0.1)

optimizer.zero_grad()
loss = w.square().sum()
loss.backward()

print("Gradient:", w.grad.item())

optimizer.step()
print("Updated weight:", w.item())

Gradient: 4.0
Updated weight: 1.899999976158142


In [4]:
# Original training dataset: no random augmentation
train_dataset = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# Fixed train/validation split
train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# Separate dataset object so augmentation doesn't affect validation
augmented_dataset = copy(train_dataset)
augmented_dataset.transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

augmented_train_subset = Subset(
    augmented_dataset,
    train_subset.indices
)

# Separate test split: no random augmentation
test_dataset = datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

# Use this loader for parameter updates
augmented_train_loader = DataLoader(
    augmented_train_subset,
    batch_size=batch_size,
    shuffle=True
)

# Use these loaders for evaluation
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"Training:   {len(train_subset):,}")
print(f"Validation: {len(val_subset):,}")
print(f"Test:       {len(test_dataset):,}")

Training:   45,000
Validation: 5,000
Test:       10,000


In [5]:
optimizer = Adam(model.parameters(), learning_rate=0.001)

train_loss = train_one_epoch(
    model, augmented_train_loader, optimizer, device
)

val_metrics = evaluate(model, val_loader, device)

print(f"Training loss: {train_loss:.4f}")
print(f"Validation loss: {val_metrics['loss']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")

Training loss: 1.5839
Validation loss: 1.6368
Validation accuracy: 46.78%


In [6]:
train_metrics = evaluate(model, train_loader, device)

print(f"Training evaluation loss: {train_metrics['loss']:.4f}")
print(f"Training evaluation accuracy: {train_metrics['accuracy']:.2f}%")

Training evaluation loss: 1.5957
Training evaluation accuracy: 47.47%


In [7]:
def take_step(w, optimizer):
    optimizer.zero_grad()
    loss = w.square().sum()
    loss.backward()
    optimizer.step()


# Train for three updates
w1 = torch.tensor([2.0], device=device, requires_grad=True)
opt1 = Adam([w1], learning_rate=0.1)

for _ in range(3):
    take_step(w1, opt1)

# Save the weight and optimizer together
saved_weight = w1.detach().clone()
saved_optimizer = opt1.state_dict()

# Recreate and restore
w2 = saved_weight.clone().requires_grad_(True)
opt2 = Adam([w2], learning_rate=0.001)
opt2.load_state_dict(saved_optimizer)

# Both take the same next step
take_step(w1, opt1)
take_step(w2, opt2)

print("Weights match:", torch.allclose(w1, w2))
print("First moments match:", torch.allclose(
    opt1.first_moments[0], opt2.first_moments[0]
))
print("Second moments match:", torch.allclose(
    opt1.second_moments[0], opt2.second_moments[0]
))
print("Step counters:", opt1.t, opt2.t)
print("Restored learning rate:", opt2.learning_rate)

Weights match: True
First moments match: True
Second moments match: True
Step counters: 4 4
Restored learning rate: 0.1


In [8]:
# Capture state before saving
state_before = model.state_dict()
step_before = optimizer.t

save_checkpoint(
    "checkpoint_helper_test.pt",
    model,
    optimizer=optimizer,
    epoch=1
)

completed_epochs = load_checkpoint(
    "checkpoint_helper_test.pt",
    model,
    optimizer=optimizer,
    device=device
)

state_after = model.state_dict()

print("Model state matches:", all(
    torch.equal(state_before[name], state_after[name])
    for name in state_before
))
print("Adam step matches:", optimizer.t == step_before)
print("Completed epochs:", completed_epochs)

Model state matches: True
Adam step matches: True
Completed epochs: 1


In [10]:
# Create once; preserve this optimizer across epochs
optimizer = Adam(model.parameters(), learning_rate=0.001)

best_val = float("-inf")

for epoch in range(1, 11):
    train_loss = train_one_epoch(
        model, augmented_train_loader, optimizer, device
    )
    metrics = evaluate(model, val_loader, device)

    if metrics["accuracy"] > best_val:
        best_val = metrics["accuracy"]
        save_checkpoint(
            "best_model.pt",
            model,
            optimizer=optimizer,
            epoch=epoch
        )

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {metrics['loss']:.4f} | "
        f"Val accuracy: {metrics['accuracy']:.2f}%"
    )

Epoch 1 | Train loss: 1.5494 | Val loss: 1.4301 | Val accuracy: 48.88%
Epoch 2 | Train loss: 1.2788 | Val loss: 1.3003 | Val accuracy: 54.84%
Epoch 3 | Train loss: 1.1910 | Val loss: 1.1742 | Val accuracy: 59.26%
Epoch 4 | Train loss: 1.1375 | Val loss: 1.1456 | Val accuracy: 60.16%
Epoch 5 | Train loss: 1.0891 | Val loss: 1.1781 | Val accuracy: 58.14%
Epoch 6 | Train loss: 1.0580 | Val loss: 1.0672 | Val accuracy: 62.52%
Epoch 7 | Train loss: 1.0297 | Val loss: 1.1042 | Val accuracy: 61.62%
Epoch 8 | Train loss: 1.0042 | Val loss: 1.0941 | Val accuracy: 62.14%
Epoch 9 | Train loss: 0.9776 | Val loss: 1.0902 | Val accuracy: 62.56%
Epoch 10 | Train loss: 0.9611 | Val loss: 1.0259 | Val accuracy: 64.54%
